In [1]:
import sys
import os

# Move one directory up (to project root)
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("Added to PYTHONPATH:", project_root)

Added to PYTHONPATH: /vols/cms/mm1221/geant4sim/scripts/validation_new


In [2]:
import pandas as pd
import glob
import os
from pathlib import Path

def load_csv_folder_with_event_offset(folder, event_col="event_id", verbose=True):
    """
    Recursively loads ALL CSVs inside `folder` and subfolders.
    Applies a running offset to event_id so events remain unique across files.
    """

    # --- recursively find csv files ---
    pattern = os.path.join(folder, "**", "*.csv")
    csv_files = glob.glob(pattern, recursive=True)

    if len(csv_files) == 0:
        raise RuntimeError(f"No CSV files found anywhere under {folder}")

    # --- sort in stable, natural path order ---
    csv_files = sorted(csv_files, key=lambda p: Path(p).as_posix())

    if verbose:
        print(f"Found {len(csv_files)} csv files")

    dfs = []
    event_offset = 0
    total_events = 0

    for i, f in enumerate(csv_files):
        if verbose and (i % 20 == 0):
            print(f"[{i+1}/{len(csv_files)}] Loading: {f}")

        df = pd.read_csv(f)

        if event_col not in df.columns:
            raise ValueError(f"{event_col} not found in {f}")

        # --- count events BEFORE offset ---
        n_events = df[event_col].nunique()

        # --- apply offset ---
        df[event_col] += event_offset

        dfs.append(df)

        event_offset += n_events
        total_events += n_events

    if verbose:
        print(f"Total unique events loaded: {total_events}")
        print(f"Final event_id range: 0 → {event_offset-1}")

    return pd.concat(dfs, ignore_index=True)

In [20]:
df_cont_em_16 = load_csv_folder_with_event_offset("../dfs_oc_16_em")
df_cont_em_3 = load_csv_folder_with_event_offset("../dfs_oc_3_em")

df_cont_had_16          = load_csv_folder_with_event_offset("../dfs_oc_16_had")
df_cont_had_3          = load_csv_folder_with_event_offset("../dfs_oc_3_had")

#df_oc_em          = load_csv_folder_with_event_offset("../dfs_table/EM_cont_16")
#df_oc_had = load_csv_folder_with_event_offset("../dfs_table/EM_cont_16")

#df_oc_Ag = df_oc

#print(df_contrastive["event_id"].min(), df_contrastive["event_id"].max())
#print(df_oc["event_id"].min(), df_oc["event_id"].max())
#print(df_oc_Ag["event_id"].min(), df_oc_Ag["event_id"].max())
#print(df_oc_noise["event_id"].min(), df_oc_noise["event_id"].max())

Found 8 csv files
[1/8] Loading: ../dfs_oc_16_em/Test_pi.1001_1002.csv
Total unique events loaded: 200
Final event_id range: 0 → 199
Found 8 csv files
[1/8] Loading: ../dfs_oc_3_em/Test_pi.1001_1002.csv
Total unique events loaded: 200
Final event_id range: 0 → 199
Found 8 csv files
[1/8] Loading: ../dfs_oc_16_had/Test_pi.1001_1002.csv
Total unique events loaded: 200
Final event_id range: 0 → 199
Found 8 csv files
[1/8] Loading: ../dfs_oc_3_had/Test_pi.1001_1002.csv
Total unique events loaded: 200
Final event_id range: 0 → 199


In [12]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph, radius_neighbors_graph

import pandas as pd
from tqdm import tqdm

def reco_to_sim(hit_energies, rmask, cmask, purities):
    num = 0
    denom = 0
    for i in range(len(hit_energies)):
        if rmask[i]:
            if cmask[i]:
                num += min((1-purities[i])**2, 1) * (hit_energies[i])**2
                denom += hit_energies[i]**2
            else:
                num +=(hit_energies[i])**2
                denom += hit_energies[i]**2
    RtS = num/denom
    return RtS


def build_dataframe(reconstructed_label, loader):
    num_reco = len(reconstructed_label)
    rows = []
    for i, data in enumerate(loader):
        if i >= num_reco:
            break
        CP_ids = data.assoc
        PrimaryEnergies = data.PrimaryEnergies
        reco_ids = reconstructed_label[i]

        hit_energy = np.array(data.x[:,3])
        hit_purity = np.ones_like(hit_energy)
        unique_cp_ids = np.unique(CP_ids)
        unique_reco_ids = np.unique(reco_ids)

        for rid in unique_reco_ids:
            if rid == -1:
                continue
            rmask = np.array((reco_ids == rid))
            for cid in unique_cp_ids:
                cmask = np.array((CP_ids == cid))
                PE = PrimaryEnergies[cid]
                
                cp_energy = hit_energy[cmask].sum()
                reco_energy = hit_energy[rmask].sum()
                shared_energy = hit_energy[rmask & cmask].sum()
                RtS = reco_to_sim(hit_energy, rmask, cmask, hit_purity)
                rows.append({
                    'event_id': i,
                    'cp_id': cid,
                    'reco_id': rid,
                    'cp_energy': cp_energy,
                    'reco_energy': reco_energy,
                    'shared_energy': shared_energy,
                    'RtS': RtS,
                    'PrimaryEnergy' : PE.item()
                })
    df = pd.DataFrame(rows).sort_values(['event_id', 'cp_id', 'reco_id']).reset_index(drop=True)
    return df


    return None

def calc_purity(df, threshold=0.2):
    total_reco = (
        df[['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )
    associated_reco = (
        df.loc[df['RtS'] < threshold, ['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )
    purity = associated_reco / total_reco if total_reco > 0 else 0.0
    return purity

def calc_efficiency(df, threshold=0.7):
    total_cp = (
        df[['event_id', 'cp_id']]
        .drop_duplicates()
        .shape[0]
    )
    associated_cp = (
        df.loc[(df['shared_energy']/df['cp_energy']) > threshold, ['event_id', 'cp_id']]
        .drop_duplicates()
        .shape[0]
    )
    efficiency = associated_cp / total_cp if total_cp > 0 else 0.0
    return efficiency

def calc_merge_rate(df, threshold=0.2):
    # Total number of reconstructed objects
    total_reco = (
        df[['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )

    if total_reco == 0:
        return 0.0

    # Count how many CPs each reco object is associated with
    reco_cp_counts = (
        df.loc[df['RtS'] < threshold, ['event_id', 'reco_id', 'cp_id']]
        .drop_duplicates()
        .groupby(['event_id', 'reco_id'])['cp_id']
        .nunique()
    )

    # Reco objects associated with more than one CP
    merged_reco = (reco_cp_counts > 1).sum()

    merge_rate = merged_reco / total_reco
    return merge_rate

def calc_response(df):
    idx_best = df.groupby(['event_id', 'cp_id'])['shared_energy'].idxmax()
    best_matches = df.loc[idx_best].copy()

    best_matches = best_matches[best_matches['cp_energy'] > 0]
    best_matches['response'] = (
        best_matches['reco_energy'] / best_matches['cp_energy']
    )

    mean_resp = best_matches['response'].mean()
    std_resp  = best_matches['response'].std(ddof=1) 

    return mean_resp, std_resp

def calc_ratio(df):
    total_reco = (
        df[['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )
    
    total_cp = (
        df[['event_id', 'cp_id']]
        .drop_duplicates()
        .shape[0]
    )
    
    ratio = total_reco / total_cp if total_cp > 0 else 0.0
    return ratio



In [21]:
df = df_cont_em_16
pur = calc_purity(df)
eff = calc_efficiency(df)
rati = calc_ratio(df)

In [22]:
print(pur)
print(eff)
print(rati)

0.6170763260025873
0.8676646706586826
0.694311377245509
